In [1]:
import os, sys
sys.path.append(os.getenv("FALCON_PATH"))
print("sys.path:", sys.path)

sys.path: ['/opt/homebrew/Cellar/python@3.14/3.14.2/Frameworks/Python.framework/Versions/3.14/lib/python314.zip', '/opt/homebrew/Cellar/python@3.14/3.14.2/Frameworks/Python.framework/Versions/3.14/lib/python3.14', '/opt/homebrew/Cellar/python@3.14/3.14.2/Frameworks/Python.framework/Versions/3.14/lib/python3.14/lib-dynload', '', '/Users/ajayumashankar25gmail.com/Desktop/CAPSTONE/implementation/venv/lib/python3.14/site-packages', '/opt/homebrew/opt/python-tk@3.14/libexec', '/Users/ajayumashankar25gmail.com/Desktop/CAPSTONE/implementation/falcon_py']


In [2]:
import json, time
from falcon import Falcon

model = Falcon(512)

In [10]:
def serialize_command(command):
    return json.dumps(
        command, separators=(":", ","),
        sort_keys=True
    ).encode('utf-8')

In [11]:
class Drone:
    def __init__(self, drone_id):
        self.drone_id = drone_id
        self.secret_key, self.public_key = model.keygen()

    def sign_command(self, command):
        serialized = serialize_command(command)
        signature = model.sign(self.secret_key, serialized)
        return signature

In [12]:
class TrustRegistry:
    def __init__(self):
        self.registry = {}

    def register_drone(self, drone):
        self.registry[drone.drone_id] = drone

    def verify(self, drone_id, command, signature):
        if drone_id not in self.registry:
            raise Exception("Drone not registered")
        
        drone = self.registry[drone_id]
        public_key = drone.public_key
        serialized = serialize_command(command)
        return model.verify(public_key, serialized, signature)

In [13]:
def threshold_authorize(registry:dict, command: dict, signatures: dict, k:int) -> bool:
    valid_signatures = 0

    for drone_id, sign in signatures.items():
        try:
            registry.verify(drone_id, command, sign)
            valid_signatures += 1
        
        except Exception as e:
            print(f"Verification failed for drone {drone_id}: {e}")

    return valid_signatures >= k

In [18]:
# Create drones
d1 = Drone("D1")
d2 = Drone("D2")
d3 = Drone("D3")
attacker_drone = Drone("attacker")

registry = TrustRegistry()
registry.register_drone(d1)
registry.register_drone(d2)
registry.register_drone(d3)

command = {
    "command_id": "ATTACK",
    "timestamp": 123456789,
    "payload": {"target": "ZONE_A"}
}

In [19]:
signatures = {
    d1.drone_id: d1.sign_command(command),
    d2.drone_id: d2.sign_command(command)
}

try:
    if threshold_authorize(registry, command, signatures, k=2):
        print("Command authorized with 2 signatures")
    else:
        print("Command NOT authorized with 2 signatures")
except Exception as e:
    print("Error during authorization:", e)

Command authorized with 2 signatures


In [20]:
signatures = {
    d1.drone_id: d1.sign_command(command),
}

try:
    if threshold_authorize(registry, command, signatures, k=2):
        print("Command authorized with 2 signatures")
    else:
        print("Command NOT authorized with 2 signatures")
except Exception as e:
    print("Error during authorization:", e)

Command NOT authorized with 2 signatures


In [21]:
signatures = {
    d1.drone_id: d1.sign_command(command),
    attacker_drone.drone_id: attacker_drone.sign_command(command)
}

try:
    if threshold_authorize(registry, command, signatures, k=2):
        print("Command authorized with 2 signatures")
    else:
        print("Command NOT authorized with 2 signatures")
except Exception as e:
    print("Error during authorization:", e)

Verification failed for drone attacker: Drone not registered
Command NOT authorized with 2 signatures
